In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# --- 1. SUPPRESS WARNINGS ---
warnings.filterwarnings("ignore")
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)  

# --- 2. DEFINE DIRECTORIES & NEW REPO PATHS ---
WORKING_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/"
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
sc.settings.set_figure_params(dpi=300, dpi_save=300)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['figure.titleweight'] = 'bold'
mpl.rcParams['axes.grid'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
sns.set_style("ticks", {"axes.grid": False})

COND_ORDER = [
    "Control", "Prematurely-terminated", "3month-protocol", 
    "6month-protocol", "PCMV/PRV", "Rejection"
]

COND_PALETTE = {
    "Control": "#EFE48E", "3month-protocol": "skyblue", 
    "6month-protocol": "#08528f", "Prematurely-terminated": "#C9B8D9", 
    "PCMV/PRV": "#F0C096", "Rejection": "#E09E9E"
}


In [3]:
adata = sc.read ("/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad")

In [3]:
# --- 1. NORMALIZE RAW UMIs FOR SCORING ---

adata_gss = adata.copy()
sc.pp.normalize_total(adata_gss, target_sum=1e4)
sc.pp.log1p(adata_gss)


In [11]:
# --- 2. BIOLOGICALLY CURATED PATHWAYS DICTIONARY ---
pathways = {
    "FAO": [
        
    # uptake and activation
    "CD36", "SLC27A1", 
    # carnitine shuttle (MLYCD removes malonyl-CoA, the brake on CPT1)
    "CPT1A", "CPT1B", "SLC25A20", "CPT2", "MLYCD",
    # beta-oxidation spiral
    "ACADVL", "ACADL", "ACADM", "ACADS",
        # "HADHA", "HADHB",
        "ECHS1", "HADH", "ACAA2",
    # unsaturated fatty acids
    "ECI1", "DECR1",
    # electron transfer to the respiratory chain
    "ETFA", "ETFB", "ETFDH" 
        
    ],
    "Ketone_metabolism": ["BDH1", "BDH2", 
                         # "HMGCS2", 
                          "SLC16A6", "OXCT1", "ACAT1", "SLC16A1", "SLC16A7"],
    "Lactate_metabolism": ["LDHA", "LDHB", "SLC16A1", "SLC16A3", "SIRT1", "HIF1A"],
    "Glycolysis": ["SLC2A4", "HK1", "HK2", "PFKM", "PFKFB3", "PKM", "GPI",
                   # "ALDOA",
                   "PDHA1", "PDK1", "PFKL", "LDHB", "ENO1", "ENO2"],
    "Carbohydrate_metabolism": [
        # "G6PC", "SLC2A2","FBP1",
        # "PYGL", "PYGM", "GCK", "PDK4",  "SLC2A5",  "GPI"
            "UGP2", "GYS1", "GYG1", "GBE1", "PPP1R3A", "PPP1R3C",
    # breakdown (glycogenolysis)
    "PYGB", "PYGM", "PYGL", "AGL", "PHKB",
    
    
    ],
    "Amino_acid_metabolism": ["GLS", "GLUD1", "SLC1A5", "SLC38A1", "ASNS", "BCAT2", 
                              "CTH", "SHMT1",
                              
                              # "GPT2", "PHGDH", "PSAT1",
                              "GOT1", "GOT2", "SLC7A11", "BCAT1"],
    "Metabolic_dysfunction": ["PDK4", "HIF1A",  "LDHA", "SREBF1", "FABP4", "PRKAB1", "TXNRD1", "HMOX1", "UCP2", "UCP3"],
    "TCA_OXPHOS": ["PDHA1", "DLAT", "CS", "ACO2", "IDH3A", "OGDH", "SUCLG1",  "MDH2", "NDUFA9", "NDUFS1", "SDHB", "UQCRC1",  "COX5A", "ATP5F1A", "TFAM", "VDAC1",
    #                # "ATP5F1B"
                  ]
}

# --- 3. FILTER GENES BY EXPRESSION THRESHOLD (>= 0.0125) ---
print("\n--- Filtering Genes by Expression Threshold (0.0125) ---")
threshold = 0.0125
filtered_pathways = {}

for pathway_name, gene_list in pathways.items():
    # Only look at genes actually present in the dataset
    genes_in_data = [g for g in gene_list if g in adata_gss.var_names]
    
    if not genes_in_data:
        print(f"[{pathway_name}] No genes found in dataset. Skipping.")
        continue
        
    # Efficiently get expression data for these genes and group by Condition
    expr_df = sc.get.obs_df(adata_gss, keys=genes_in_data + ['Condition'])
    
    # Calculate the mean expression per condition, then find the max mean across conditions for each gene
    max_mean_expr = expr_df.groupby('Condition', observed=True).mean().max()
    
    # Separate genes based on the threshold
    retained_genes = max_mean_expr[max_mean_expr >= threshold].index.tolist()
    removed_genes = max_mean_expr[max_mean_expr < threshold].index.tolist()
    
    # Print reporting
    if removed_genes:
        print(f"[{pathway_name}] Removed ({len(removed_genes)}): {', '.join(removed_genes)}")
    else:
        print(f"[{pathway_name}] No genes removed.")
        
    # Only keep the retained genes if there's at least one
    if retained_genes:
        filtered_pathways[pathway_name] = retained_genes

# Replace old pathways dictionary with the newly filtered one
pathways = filtered_pathways
print("-------------------------------------------------------\n")


--- Filtering Genes by Expression Threshold (0.0125) ---


[FAO] Removed (2): ECI1, ETFB


[Ketone_metabolism] No genes removed.


[Lactate_metabolism] No genes removed.


[Glycolysis] No genes removed.


[Carbohydrate_metabolism] No genes removed.


[Amino_acid_metabolism] No genes removed.


[Metabolic_dysfunction] No genes removed.


[TCA_OXPHOS] No genes removed.
-------------------------------------------------------



In [12]:
# --- 4. SCORE GENES USING FILTERED LISTS ---
print("Scoring pathways...")
for pathway_name, gene_list in pathways.items():
    sc.tl.score_genes(adata_gss, gene_list=gene_list, score_name=f"{pathway_name}_score")

# Save the scored object
gss_path = os.path.join(WORKING_DIR, 'global_metabolic_gssa_preclinical.h5ad')
adata_gss.write(gss_path)
print(f"Saved scored AnnData to {gss_path}")


Scoring pathways...


Saved scored AnnData to /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/global_metabolic_gssa_preclinical.h5ad


In [13]:
adata_gss = sc.read ('/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/global_metabolic_gssa_preclinical.h5ad')


# Pathway Overlap & Global Aggregations

In [14]:
# --- SET METADATA COLUMNS ---
condition_col = "Condition" 
sample_col = "Sample_ID"
# All tables are written by the export cell at the end of the notebook.

# --- 1. GENE OVERLAP HEATMAP ---
print("Calculating pathway overlap...")
pathway_names = list(pathways.keys())
overlap_matrix = np.zeros((len(pathway_names), len(pathway_names)))

for i, p1 in enumerate(pathway_names):
    for j, p2 in enumerate(pathway_names):
        if i <= j:
            shared = len(set(pathways[p1]).intersection(set(pathways[p2])))
            overlap_matrix[i, j] = shared
            overlap_matrix[j, i] = shared

overlap_df = pd.DataFrame(overlap_matrix, index=pathway_names, columns=pathway_names)

plt.figure(figsize=(4, 3))
sns.heatmap(overlap_df, annot=True, cmap="Blues", fmt=".0f", linewidths=0.5, 
            annot_kws={"weight": "bold", "size": 5})
plt.title("METABOLIC PATHWAYS GENE OVERLAP", fontweight='bold', fontsize=5)
plt.xticks(rotation=45, ha='right', fontweight='bold', fontsize=5)
plt.yticks(rotation=0, fontweight='bold', fontsize=5)
plt.tight_layout()
plt.savefig("Global_Pathway_Gene_Overlap.pdf", format='pdf', dpi=300, bbox_inches='tight')
plt.close()

# --- 2. AGGREGATE SCORES BY SAMPLE & CONDITION ---
print("Aggregating global scores...")
score_cols = [f"{p}_score" for p in pathways.keys() if f"{p}_score" in adata_gss.obs.columns]
df_cells = adata_gss.obs[score_cols + [sample_col, condition_col]].copy()

# Median per Sample
df_sample = df_cells.groupby(sample_col, observed=True)[score_cols].median()
sample_to_cond = df_cells.drop_duplicates(sample_col).set_index(sample_col)[condition_col]
df_sample[condition_col] = sample_to_cond.loc[df_sample.index]


# Median per Condition
df_condition = df_sample.groupby(condition_col, observed=True)[score_cols].median()
# Reorder safely to match COND_ORDER
df_condition = df_condition.reindex([c for c in COND_ORDER if c in df_condition.index])


Calculating pathway overlap...


Aggregating global scores...


# Global Boxplots & Pearson Correlation

In [15]:
# --- 1. GLOBAL BOXPLOT BY CONDITION ---
print("Generating global boxplots...")
df_melted = df_sample.melt(id_vars=condition_col, value_vars=score_cols, var_name='Pathway', value_name='Score')
df_melted['Pathway'] = df_melted['Pathway'].str.replace("_score", "")

plt.figure(figsize=(6, 3))
sns.boxplot(data=df_melted, x='Pathway', y='Score', hue=condition_col, 
            hue_order=[c for c in COND_ORDER if c in df_melted[condition_col].unique()], 
            palette=COND_PALETTE, linewidth=0.5, fliersize=1)

plt.title("GENE SET SCORES BY PATHWAY AND CONDITION", fontweight='bold', fontsize=5)
plt.xticks(rotation=30, ha='right', fontweight='bold', fontsize=5)
plt.yticks(fontweight='bold', fontsize=5)
plt.xlabel("PATHWAY", fontweight='bold', fontsize=5)
plt.ylabel("SCORE", fontweight='bold', fontsize=5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, prop={'weight':'bold', 'size':5})
plt.tight_layout()
plt.savefig("Global_Scores_Boxplot.pdf", format='pdf', dpi=300, bbox_inches='tight')
plt.close()

# --- 2. PEARSON CORRELATION (Condition Level) ---
print("Generating Pearson correlation matrix...")
corr_mat = df_condition.T.corr(method='pearson')

plt.figure(figsize=(4, 3))
sns.heatmap(corr_mat, cmap='RdBu_r', annot=True, fmt=".2f", linewidths=0.5, square=True, 
            annot_kws={"weight": "bold", "size": 5})
plt.title("METABOLIC PATHWAYS PEARSON CORRELATION", fontweight='bold', fontsize=5)
plt.xticks(rotation=30, ha='right', fontweight='bold', fontsize=5)
plt.yticks(rotation=0, fontweight='bold', fontsize=5)
plt.tight_layout()
plt.savefig("Global_Condition_Pearson_Correlation.pdf", format='pdf', dpi=300, bbox_inches='tight')
plt.close()


Generating global boxplots...


Generating Pearson correlation matrix...


# Cell_type analysis loop: Boxplots

In [16]:

cell_type_col = "cell_type" 

def run_highlevel_celltype_scoring(adata, pathways, ct_col, samp_col, cond_col, cond_order, cond_palette, outdir):
    Path(outdir).mkdir(parents=True, exist_ok=True)
    score_cols = [f"{p}_score" for p in pathways.keys() if f"{p}_score" in adata.obs.columns]
    cell_types = sorted(adata.obs[ct_col].dropna().unique().astype(str).tolist())
    
    print(f"\n--- Running Per-Cell Type Analysis ({len(cell_types)} clusters) ---")
    
    for ct in cell_types:
        safe_ct = "".join(c if c.isalnum() else "_" for c in ct)
        ct_dir = Path(outdir) / safe_ct
        ct_dir.mkdir(parents=True, exist_ok=True)
        
        ad_ct = adata[adata.obs[ct_col] == ct].copy()
        if ad_ct.n_obs == 0: continue
            
        df = ad_ct.obs[score_cols + [samp_col, cond_col]].copy()
        df = df.rename(columns={c: c.replace("_score", "") for c in score_cols})
        p_names = [c.replace("_score", "") for c in score_cols]
        
        # Aggregate per Sample (Median)
        sample_scores = df.groupby(samp_col, observed=True)[p_names].median()
        sample_to_cond = df.drop_duplicates(samp_col).set_index(samp_col)[cond_col]
        sample_scores[cond_col] = sample_to_cond.loc[sample_scores.index]
        
        # Melt for plotting
        present_conds = [c for c in cond_order if c in sample_scores[cond_col].unique()]
        melted = sample_scores.melt(id_vars=[cond_col], value_vars=p_names, var_name="Pathway", value_name="Score")
        melted = melted[melted[cond_col].isin(present_conds)].copy()
        melted[cond_col] = pd.Categorical(melted[cond_col], categories=present_conds, ordered=True)
        
        # 1. BOXPLOT
        plt.figure(figsize=(6, 3))
        sns.boxplot(data=melted, x="Pathway", y="Score", hue=cond_col, hue_order=present_conds, 
                    palette=cond_palette, linewidth=0.5, fliersize=1)
        plt.title(f"{ct.upper()} - METABOLIC SCORES", fontweight='bold', fontsize=5)
        plt.xticks(rotation=30, ha="right", fontweight='bold', fontsize=5)
        plt.yticks(fontweight='bold', fontsize=5)
        plt.xlabel("PATHWAY", fontweight='bold', fontsize=5)
        plt.ylabel("SCORE", fontweight='bold', fontsize=5)
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False, prop={'weight':'bold', 'size':5})
        plt.tight_layout()
        plt.savefig(ct_dir / f"{safe_ct}_Boxplot.pdf", format='pdf', dpi=300, bbox_inches='tight')
        plt.close()

        # 2. CONDITION HEATMAP
        heatmap_df = melted.groupby([cond_col, "Pathway"], observed=True)["Score"].mean().unstack("Pathway")
        
        plt.figure(figsize=(4, 3))
        sns.heatmap(heatmap_df, cmap="RdBu_r", annot=True, fmt=".2f", linewidths=0.5, 
                    annot_kws={"weight": "bold", "size": 5})
        plt.title(f"{ct.upper()} - MEAN PATHWAY SCORES", fontweight='bold', fontsize=5)
        plt.xticks(rotation=30, ha="right", fontweight='bold', fontsize=5)
        plt.yticks(rotation=0, fontweight='bold', fontsize=5)
        plt.tight_layout()
        plt.savefig(ct_dir / f"{safe_ct}_Heatmap.pdf", format='pdf', dpi=300, bbox_inches='tight')
        plt.close()

    print(f"Per-celltype analysis complete. Results saved in {outdir}")

# --- EXECUTE THE LOOP ---
if cell_type_col in adata_gss.obs.columns:
    run_highlevel_celltype_scoring(
        adata=adata_gss,
        pathways=pathways,
        ct_col=cell_type_col,
        samp_col=sample_col,
        cond_col=condition_col,
        cond_order=COND_ORDER,
        cond_palette=COND_PALETTE,
        outdir=os.path.join(WORKING_DIR, "Per_Celltype_Results")
    )
else:
    print(f"Update cell_type_col. '{cell_type_col}' not found in adata.obs.")
    


--- Running Per-Cell Type Analysis (10 clusters) ---


Per-celltype analysis complete. Results saved in /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/Per_Celltype_Results


# Cell_type analysis loop: Boxplots with significance with Mann Whitney U test & BH FDR adjustment

In [17]:
import os
import warnings
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# --- 1. SUPPRESS WARNINGS ---
warnings.filterwarnings("ignore")

# --- 2. ENFORCE AESTHETICS ---
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['figure.titleweight'] = 'bold'
mpl.rcParams['axes.grid'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
sns.set_style("ticks", {"axes.grid": False})

# --- 3. DYNAMIC CONFIGURATION (Change this anytime) ---
REF_COND = "3month-protocol"  # Change to '3month-protocol' or any other condition when needed

# Format directory name properly
ref_dir_name = "Ref_" + REF_COND.replace("-protocol", "").replace("/", "_")
BASE_SIG_DIR = f"/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/celltype_boxplots_significance/{ref_dir_name}/"
os.makedirs(BASE_SIG_DIR, exist_ok=True)

condition_col = "Condition" 
sample_col = "Sample_ID"
cell_type_col = "cell_type"

# --- 4. SIGNIFICANCE ANNOTATION PARAMETERS (Highly Replicable/Adjustable) ---
# These values are percentages of the plot's total height (e.g., 0.05 = 5% of the plot height)
BRACKET_START_SPACING = 0.05    # Space between the tallest boxplot point and the first bracket
BRACKET_TICK_HEIGHT = 0.03      # The height of the vertical "legs" of the bracket
BRACKET_STEP = 0.12             # Space between stacked brackets (if multiple are significant)
STAR_TEXT_SPACING = 0.01        # Space between the horizontal bracket line and the stars
STAR_FONT_SIZE = 7              # Font size specifically for the asterisks
BRACKET_LINE_WIDTH = 0.8        # Thickness of the bracket lines

print(f"Reference Condition set to: {REF_COND}")
print(f"Outputs will be saved to: {BASE_SIG_DIR}")

# --- 5. STATISTICAL PIPELINE & PLOTTING ---
def run_detailed_stats_and_plotting(adata, pathways, ct_col, samp_col, cond_col, ref_cond, cond_order, cond_palette, outdir):
    score_cols = [f"{p}_score" for p in pathways.keys() if f"{p}_score" in adata.obs.columns]
    p_names = [c.replace("_score", "") for c in score_cols]
    cell_types = sorted(adata.obs[ct_col].dropna().unique().astype(str).tolist())
    
    print(f"\n--- Running Detailed Stats & Plotting ({len(cell_types)} clusters) ---")
    
    for ct in cell_types:
        safe_ct = "".join(c if c.isalnum() else "_" for c in ct)
        ct_dir = Path(outdir) / safe_ct
        ct_dir.mkdir(parents=True, exist_ok=True)
        
        ad_ct = adata[adata.obs[ct_col] == ct].copy()
        if ad_ct.n_obs == 0: continue
            
        # Extract data and aggregate to per-sample median (Pseudobulking)
        df = ad_ct.obs[score_cols + [samp_col, cond_col]].copy()
        df = df.rename(columns={c: c.replace("_score", "") for c in score_cols})
        
        sample_scores = df.groupby(samp_col, observed=True)[p_names].median()
        sample_to_cond = df.drop_duplicates(samp_col).set_index(samp_col)[cond_col]
        sample_scores[cond_col] = sample_to_cond.loc[sample_scores.index]
        
        # Determine available conditions
        present_conds = [c for c in cond_order if c in sample_scores[cond_col].unique()]
        if ref_cond not in present_conds:
            print(f"[{ct}] Reference '{ref_cond}' missing. Skipping stats.")
            continue
            
        # --- STATISTICAL TESTING (SAMPLE LEVEL, MANN-WHITNEY U) ---
        stats_results = []
        for pathway in p_names:
            ref_data = sample_scores[sample_scores[cond_col] == ref_cond][pathway].dropna().values
            mean_ref = np.mean(ref_data) if len(ref_data) > 0 else np.nan
            
            for comp_cond in present_conds:
                if comp_cond == ref_cond: continue
                
                comp_data = sample_scores[sample_scores[cond_col] == comp_cond][pathway].dropna().values
                mean_comp = np.mean(comp_data) if len(comp_data) > 0 else np.nan
                
                pval = np.nan
                mean_diff = mean_comp - mean_ref
                
                if len(ref_data) > 0 and len(comp_data) > 0:
                    try:
                        # Mann-Whitney U test (non-parametric)
                        stat, pval = mannwhitneyu(comp_data, ref_data, alternative='two-sided')
                    except Exception:
                        pass
                
                stats_results.append({
                    'Pathway': pathway,
                    'Cell_Type': ct,
                    'Ref_condition': ref_cond,
                    'Comp_condition': comp_cond,
                    'N_Samples_Ref': len(ref_data),
                    'N_Samples_Comp': len(comp_data),
                    'Mean_Ref': mean_ref,
                    'Mean_Comp': mean_comp,
                    'Mean_Difference': mean_diff,
                    'pvalue': pval
                })
        
        # --- APPLY FDR (BH) CORRECTION (INDEPENDENTLY PER CONDITION) ---
        stats_df = pd.DataFrame(stats_results)
        stats_df['FDR'] = np.nan
        
        # Loop through each comparison condition and adjust FDR across the pathways for that specific comparison
        for comp in stats_df['Comp_condition'].unique():
            mask = (stats_df['Comp_condition'] == comp) & stats_df['pvalue'].notna()
            if mask.sum() > 0:
                _, fdr_bh, _, _ = multipletests(stats_df.loc[mask, 'pvalue'], method='fdr_bh')
                stats_df.loc[mask, 'FDR'] = fdr_bh
            
        
                
        # --- PLOTTING INDIVIDUAL BOXPLOTS WITH SIGNIFICANCE BRACKETS ---
        melted = sample_scores.melt(id_vars=[cond_col], value_vars=p_names, var_name="Pathway", value_name="Score")
        melted = melted[melted[cond_col].isin(present_conds)].copy()
        
        # Map conditions to X-axis indices
        x_positions = {c: i for i, c in enumerate(present_conds)}
        
        for pathway in p_names:
            pathway_data = melted[melted['Pathway'] == pathway]
            
            fig, ax = plt.subplots(figsize=(4, 3))
            sns.boxplot(data=pathway_data, x=cond_col, y="Score", order=present_conds, 
                        palette=cond_palette, linewidth=0.5, fliersize=1, ax=ax)
            
            # --- CALCULATE DYNAMIC SPACING FOR WHISKERS ---
            y_max = pathway_data['Score'].max()
            y_min = pathway_data['Score'].min()
            y_range = y_max - y_min if y_max != y_min else 1.0  # Prevent division by zero
            
            current_y = y_max + (BRACKET_START_SPACING * y_range)
            tick_h = BRACKET_TICK_HEIGHT * y_range
            step_y = BRACKET_STEP * y_range
            
            # Filter significant tests for this specific pathway
            sig_tests = stats_df[(stats_df['Pathway'] == pathway) & (stats_df['FDR'] < 0.05)]
            
            for _, row in sig_tests.iterrows():
                c1, c2 = row['Ref_condition'], row['Comp_condition']
                if c1 in x_positions and c2 in x_positions:
                    x1, x2 = x_positions[c1], x_positions[c2]
                    
                    # Determine stars
                    if row['FDR'] < 0.001: stars = "***"
                    elif row['FDR'] < 0.01: stars = "**"
                    else: stars = "*"
                    
                    # Draw bracket (whisker)
                    # Coordinates: [x1, x1, x2, x2], [y_start, y_top, y_top, y_end]
                    ax.plot([x1, x1, x2, x2], 
                            [current_y, current_y + tick_h, current_y + tick_h, current_y], 
                            lw=BRACKET_LINE_WIDTH, color='black')
                    
                    # Add bold stars
                    ax.text((x1 + x2) * 0.5, current_y + tick_h + (STAR_TEXT_SPACING * y_range), 
                            stars, ha='center', va='bottom', color='black', 
                            fontweight='bold', fontsize=STAR_FONT_SIZE)
                    
                    # Move the current Y up for the next stacked bracket (if any)
                    current_y += step_y
            
            # Dynamically adjust the Y-axis limit so the highest bracket doesn't get cut off
            if not sig_tests.empty:
                ax.set_ylim(y_min - (0.05 * y_range), current_y + (0.05 * y_range))
            
            # Formatting the plot
            ax.set_title(f"{ct.upper()} - {pathway.upper()}", fontweight='bold', fontsize=5)
            ax.set_xticks(range(len(present_conds)))
            ax.set_xticklabels(present_conds, rotation=45, ha="right", fontweight='bold', fontsize=5)
            plt.yticks(fontweight='bold', fontsize=5)
            ax.set_xlabel("CONDITION", fontweight='bold', fontsize=5)
            ax.set_ylabel("SCORE", fontweight='bold', fontsize=5)
            
            # Save strictly at 300 DPI PDF
            plot_path = ct_dir / f"{safe_ct}_{pathway}_boxplot.pdf"
            plt.tight_layout()
            plt.savefig(plot_path, format='pdf', dpi=300, bbox_inches='tight')
            plt.close()

    print(f"Finished! All statistical CSVs and annotated plots are saved in: {outdir}")

# --- 6. EXECUTE ---
if cell_type_col in adata_gss.obs.columns:
    run_detailed_stats_and_plotting(
        adata=adata_gss,
        pathways=pathways,
        ct_col=cell_type_col,
        samp_col=sample_col,
        cond_col=condition_col,
        ref_cond=REF_COND,
        cond_order=COND_ORDER,
        cond_palette=COND_PALETTE,
        outdir=BASE_SIG_DIR
    )
else:
    print(f"Error: '{cell_type_col}' not found in adata.obs.")

Reference Condition set to: 3month-protocol
Outputs will be saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/celltype_boxplots_significance/Ref_3month/

--- Running Detailed Stats & Plotting (10 clusters) ---


[Lymph_sus] Reference '3month-protocol' missing. Skipping stats.


Finished! All statistical CSVs and annotated plots are saved in: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/celltype_boxplots_significance/Ref_3month/


# Across all cell types: boxplots with significance (Mann Whitney U test & BH FDR adjustment)

In [18]:
# Boxplots per pathway
# the score of each pathway per heart over ALL its nuclei (every cell type pooled), drawn for every condition, with each transplanted condition tested against 3month-protocol.

import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
POOLED_DIR = os.path.join(WORKING_DIR, "across_all_celltype_boxplots_significance")
os.makedirs(POOLED_DIR, exist_ok=True)

POOLED_REF = "3month-protocol"
POOLED_COMPARISONS = ["Prematurely-terminated", "6month-protocol", "Rejection", "PCMV/PRV"]   # each against POOLED_REF
POOLED_CONDITIONS = ["Control", "Prematurely-terminated", "3month-protocol",
                     "6month-protocol", "PCMV/PRV", "Rejection"]                             # all drawn, in this order
POOLED_PALETTE = {
    "Control": "#EFE48E", "3month-protocol": "skyblue",
    "6month-protocol": "#08528f", "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV": "#F0C096", "Rejection": "#E09E9E",
}

# --- 2. VISUAL CONFIGURATION (same bracket geometry as the per-cell-type significance boxplots) ---
POOLED_FIG = dict(
    w=4, h=3,                    # figure size (inches)
    box_width=0.6,               # box width (fraction of the category spacing)
    box_lw=0.5,                  # box and whisker line width
    dot_size=3,                  # heart dots (points)
    dot_edge_lw=0.4,             # dot outline width
    jitter=0.12,                 # horizontal spread of the dots
    bracket_start=0.05,          # gap between the highest dot and the first bracket (fraction of the y range)
    bracket_tick=0.03,           # height of the bracket legs
    bracket_step=0.12,           # spacing between stacked brackets
    star_gap=0.01,               # gap between bracket and stars
    star_fs=7,                   # star font size
    bracket_lw=0.8,              # bracket line width
)

# --- 3. PER-HEART MEDIAN OVER ALL NUCLEI ---
pooled_score_cols = [f"{p}_score" for p in pathways.keys() if f"{p}_score" in adata_gss.obs.columns]
pooled_paths = [c.replace("_score", "") for c in pooled_score_cols]
cells_all = adata_gss.obs[pooled_score_cols + ["Sample_ID", "Condition"]].copy()
cells_all.columns = pooled_paths + ["Sample_ID", "Condition"]
assert cells_all.groupby("Sample_ID", observed=True)["Condition"].nunique().max() == 1, "a heart has more than one condition"

pooled_hearts = cells_all.groupby("Sample_ID", observed=True)[pooled_paths].median()
pooled_hearts["Condition"] = (cells_all.drop_duplicates("Sample_ID").set_index("Sample_ID")["Condition"]
                              .astype(str).loc[pooled_hearts.index].values)
present = [c for c in POOLED_CONDITIONS if c in set(pooled_hearts["Condition"])]
assert POOLED_REF in present, f"{POOLED_REF} has no hearts"

# --- 4. TESTS: EACH COMPARISON AGAINST 3MONTH-PROTOCOL, BH ACROSS PATHWAYS WITHIN THE COMPARISON ---
rows = []
for comp in POOLED_COMPARISONS:
    for p in pooled_paths:
        ref = pooled_hearts.loc[pooled_hearts["Condition"] == POOLED_REF, p].dropna().values
        cmp_ = pooled_hearts.loc[pooled_hearts["Condition"] == comp, p].dropna().values
        pval = mannwhitneyu(cmp_, ref, alternative="two-sided").pvalue if len(ref) and len(cmp_) else np.nan
        rows.append({"Pathway": p, "Reference": POOLED_REF, "Compared condition": comp,
                     "Hearts, reference (n)": len(ref), "Hearts, compared (n)": len(cmp_),
                     "Mean, reference": np.mean(ref), "Mean, compared": np.mean(cmp_),
                     "Difference": np.mean(cmp_) - np.mean(ref), "P": pval})
pooled_tests = pd.DataFrame(rows)
pooled_tests["FDR"] = np.nan
for comp in POOLED_COMPARISONS:
    m = (pooled_tests["Compared condition"] == comp) & pooled_tests["P"].notna()
    pooled_tests.loc[m, "FDR"] = multipletests(pooled_tests.loc[m, "P"], method="fdr_bh")[1]
pooled_tests["Mark"] = np.select([pooled_tests["FDR"] < 0.001, pooled_tests["FDR"] < 0.01, pooled_tests["FDR"] < 0.05],
                                 ["***", "**", "*"], "")

# --- 5. ONE BOXPLOT PER PATHWAY ---
x_pos = {c: i for i, c in enumerate(present)}
f = POOLED_FIG
for p in pooled_paths:
    d = pooled_hearts[["Condition", p]].rename(columns={p: "Score"})
    d = d[d["Condition"].isin(present)]
    fig, ax = plt.subplots(figsize=(f["w"], f["h"]))
    sns.boxplot(data=d, x="Condition", y="Score", order=present, hue="Condition", hue_order=present,
                palette=POOLED_PALETTE, dodge=False, legend=False, width=f["box_width"],
                linewidth=f["box_lw"], showfliers=False, ax=ax)
    sns.stripplot(data=d, x="Condition", y="Score", order=present, hue="Condition", hue_order=present,
                  palette=POOLED_PALETTE, dodge=False, legend=False, size=f["dot_size"], jitter=f["jitter"],
                  edgecolor="black", linewidth=f["dot_edge_lw"], ax=ax, zorder=3)

    y_max, y_min = d["Score"].max(), d["Score"].min()
    y_range = (y_max - y_min) or 1.0
    current_y = y_max + f["bracket_start"] * y_range
    tick_h, step_y = f["bracket_tick"] * y_range, f["bracket_step"] * y_range
    sig = pooled_tests[(pooled_tests["Pathway"] == p) & (pooled_tests["FDR"] < 0.05)].copy()
    sig["dist"] = [abs(x_pos[c] - x_pos[POOLED_REF]) for c in sig["Compared condition"]]
    for _, row in sig.sort_values("dist").iterrows():          # shortest bracket lowest
        x1, x2 = sorted([x_pos[POOLED_REF], x_pos[row["Compared condition"]]])
        ax.plot([x1, x1, x2, x2], [current_y, current_y + tick_h, current_y + tick_h, current_y],
                lw=f["bracket_lw"], color="black")
        ax.text((x1 + x2) / 2, current_y + tick_h + f["star_gap"] * y_range, row["Mark"],
                ha="center", va="bottom", color="black", fontweight="bold", fontsize=f["star_fs"])
        current_y += step_y
    ax.set_ylim(y_min - 0.05 * y_range, (current_y if len(sig) else y_max) + 0.05 * y_range)

    ax.set_title(f"ALL CELL TYPES - {p.upper()}", fontweight="bold", fontsize=5)
    ax.set_xticks(range(len(present)))
    ax.set_xticklabels(present, rotation=45, ha="right", fontweight="bold", fontsize=5)
    plt.yticks(fontweight="bold", fontsize=5)
    ax.set_xlabel("CONDITION", fontweight="bold", fontsize=5)
    ax.set_ylabel("SCORE (median per heart, all nuclei)", fontweight="bold", fontsize=5)
    plt.tight_layout()
    plt.savefig(os.path.join(POOLED_DIR, f"All_celltypes_{p}_boxplot.pdf"), format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)

print("Hearts: " + ", ".join(f"{c} {int((pooled_hearts['Condition'] == c).sum())}" for c in present))
print(f"{len(pooled_paths)} boxplots saved in {POOLED_DIR}")
print(pooled_tests.pivot(index="Pathway", columns="Compared condition", values="FDR")
      .reindex(index=pooled_paths, columns=POOLED_COMPARISONS).round(4).to_string())


Hearts: Control 4, Prematurely-terminated 3, 3month-protocol 7, 6month-protocol 4, PCMV/PRV 3, Rejection 5
8 boxplots saved in /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/across_all_celltype_boxplots_significance
Compared condition       Prematurely-terminated  6month-protocol  Rejection  PCMV/PRV
Pathway                                                                              
FAO                                      0.7619           0.1939     0.3569    0.5111
Ketone_metabolism                        0.7619           0.4202     0.2980    0.2933
Lactate_metabolism                       1.0000           0.0242     0.0404    0.5167
Glycolysis                               0.7619           0.0242     0.3232    0.2933
Carbohydrate_metabolism                  0.1333           0.9004     0.4935    0.2933
Amino_acid_metabolism                    0.7111           1.0000     1.0000    0.2933
Metabolic_dysfunction                    0.7619   

# Spider plot

In [19]:
adata_gss = sc.read ('/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/global_metabolic_gssa_preclinical.h5ad')

In [20]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from math import pi

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# --- A. Input / Output ---
OUTPUT_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- B. Which cell type(s) to plot ---
CELL_TYPES_TO_PLOT = ["CM"]          # Change to ["CM", "EC", "FB"] etc.
CELL_TYPE_COL      = "cell_type"     # Column in adata.obs
CONDITION_COL      = "Condition"     # Column in adata.obs
SAMPLE_COL         = "Sample_ID"     # Column in adata.obs

COND_ORDER = [
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
    "Control",
    "Prematurely-terminated",
]

COND_PALETTE = {
    "Control":                "#EFE48E",
    "3month-protocol":        "skyblue",
    "6month-protocol":        "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV":               "#F0C096",
    "Rejection":              "#E09E9E",
}
COND_LABELS = {
    "Control":                "Control",
    "3month-protocol":        "3-month protocol",
    "6month-protocol":        "6-month protocol",
    "Prematurely-terminated": "Preterm",
    "PCMV/PRV":               "PCMV/PRV",
    "Rejection":              "Rejection",
}


PATHWAY_DISPLAY = {
    "Lactate_metabolism":       "Lactate",
    "Ketone_metabolism":        "Ketone",
    "FAO":                      "Fatty Acid",
    "Metabolic_dysfunction":    "Metabolic\nDysfunction",
    "TCA_OXPHOS":               "TCA Cycle Mitochondrial\nFunction",
    "Amino_acid_metabolism":    "Amino Acid",
    "Carbohydrate_metabolism":  "Carbohydrate",
    "Glycolysis":               "Glycolysis/Glucose\nOxidation",
}

FIGURE_SIZE         = (8, 8)        # (width, height) in inches
RADAR_LINE_WIDTH    = 3.0           # Thickness of each condition's polygon line
RADAR_FILL_ALPHA    = 0.1           # Transparency of the filled polygon area
RADAR_DOT_SIZE      = 5             # Size of dots at each vertex
GRID_COLOR          = "#CCCCCC"     # Color of the concentric grid circles
GRID_LINE_WIDTH     = 1.0           # Thickness of grid lines
GRID_LINE_STYLE     = "-"           # "-" solid, "--" dashed, ":" dotted
SPOKE_COLOR         = "#CCCCCC"     # Color of the radial spoke lines
SPOKE_LINE_WIDTH    = 1.0           # Thickness of spoke lines
LABEL_FONT_SIZE     = 11            # Font size for pathway labels around the edge
LABEL_FONT_WEIGHT   = "bold"        # Font weight for pathway labels
LABEL_PAD           = 1.28          # Distance of labels from center (1.0 = on circle)
TICK_FONT_SIZE      = 9             # Font size for the radial tick values
TICK_FONT_COLOR     = "#333333"     # Color for tick values
TICK_BG_ALPHA       = 0.1          # Background opacity behind tick numbers (0-1)
TICK_ANGLE_DEG      = 0             # Angle (degrees) at which tick numbers are placed
                                    # 0 = top spoke; change to rotate them elsewhere
LEGEND_FONT_SIZE    = 11            # Font size for legend entries
LEGEND_LINE_WIDTH   = 3.0           # Thickness of lines in the legend
LEGEND_LOCATION     = "lower center"
LEGEND_NCOL         = 2             # Number of columns in legend
LEGEND_BBOX         = (0.5, -0.25)  # (x, y) — decrease y to push legend further down
TITLE_FONT_SIZE     = 14            # Title font size (set 0 to hide title)
DPI                 = 300           # Export resolution

# --- H. Global font settings ---
mpl.rcParams["pdf.fonttype"]    = 42
mpl.rcParams["ps.fonttype"]     = 42
mpl.rcParams["font.family"]     = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]

import scanpy as sc
sc.settings.set_figure_params(dpi=300, dpi_save=300)


# Spider plot functions

def make_spider_plot(adata, cell_type, pathway_map, cond_order, cond_palette,
                     cond_labels, ct_col, cond_col, samp_col, output_dir):
    """
    Generates a high quality radar/spider plot for one cell type.
    Saves TWO PDFs: one with tick numbers on top, one clean without numbers.
    """

    # --- Subset to cell type ---
    mask = adata.obs[ct_col].astype(str) == cell_type
    ad = adata[mask].copy()
    print(f"\n[{cell_type}] {ad.n_obs} cells")

    # --- Build score columns ---
    score_stems = list(pathway_map.keys())
    score_cols  = [f"{s}_score" for s in score_stems]
    missing = [c for c in score_cols if c not in ad.obs.columns]
    if missing:
        print(f"  [WARN] Missing score columns: {missing}")
        score_cols  = [c for c in score_cols if c in ad.obs.columns]
        score_stems = [c.replace("_score", "") for c in score_cols]

    # --- Pseudobulk: median per sample, then median per condition ---
    df = ad.obs[score_cols + [samp_col, cond_col]].copy()
    df.columns = [c.replace("_score", "") if c.endswith("_score") else c for c in df.columns]

    sample_med = df.groupby(samp_col, observed=True)[score_stems].median()
    s2c = df.drop_duplicates(samp_col).set_index(samp_col)[cond_col]
    sample_med[cond_col] = s2c.loc[sample_med.index]

    cond_med = sample_med.groupby(cond_col, observed=True)[score_stems].median()

    # Filter to requested conditions that exist
    present = [c for c in cond_order if c in cond_med.index]
    cond_med = cond_med.loc[present]

    # Reorder columns to match pathway_map order
    ordered_stems = [s for s in score_stems if s in cond_med.columns]
    cond_med = cond_med[ordered_stems]

    # --- Radar geometry ---
    N = len(ordered_stems)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]

    labels = [pathway_map.get(s, s) for s in ordered_stems]

    # --- Radial tick values (auto from data range) ---
    all_vals = cond_med.values.flatten()
    v_min = np.floor(all_vals.min() * 20) / 20
    v_max = np.ceil(all_vals.max() * 20) / 20
    n_ticks = 5
    tick_vals = np.linspace(v_min, v_max, n_ticks)


    def _build_radar(show_ticks: bool):
        fig, ax = plt.subplots(figsize=FIGURE_SIZE, subplot_kw=dict(polar=True))

        ax.set_theta_offset(pi / 2)
        ax.set_theta_direction(-1)

        # Pathway labels at each spoke
        # Pathway labels at each spoke
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, fontsize=LABEL_FONT_SIZE, fontweight=LABEL_FONT_WEIGHT)
        
        ax.tick_params(axis='x', pad=15)

        # Radial axis
        ax.set_ylim(v_min - 0.02, v_max + 0.02)
        ax.set_yticks(tick_vals)
        ax.set_yticklabels([])  # always hide default labels; we draw our own

        # Grid styling
        ax.yaxis.grid(True, color=GRID_COLOR, linewidth=GRID_LINE_WIDTH,
                       linestyle=GRID_LINE_STYLE)
        ax.xaxis.grid(True, color=SPOKE_COLOR, linewidth=SPOKE_LINE_WIDTH)
        ax.spines["polar"].set_visible(False)

        # --- Plot each condition polygon ---
        for cond in present:
            values = cond_med.loc[cond].values.tolist()
            values += values[:1]

            color = cond_palette.get(cond, "gray")
            lbl   = cond_labels.get(cond, cond)

            ax.plot(angles, values,
                    linewidth=RADAR_LINE_WIDTH, color=color, label=lbl, zorder=3)
            ax.fill(angles, values,
                    alpha=RADAR_FILL_ALPHA, color=color, zorder=2)
            ax.scatter(angles[:-1], values[:-1],
                       s=RADAR_DOT_SIZE, color=color, zorder=4)

      
        tick_texts = []
        if show_ticks:
            tick_angle_rad = np.deg2rad(TICK_ANGLE_DEG)
            for tv in tick_vals:
                t = ax.text(
                    tick_angle_rad, tv, f"{tv:.2f}",
                    fontsize=TICK_FONT_SIZE,
                    color=TICK_FONT_COLOR,
                    ha="center", va="center",
                    zorder=15,
                    bbox=dict(facecolor="white", edgecolor="none",
                              alpha=TICK_BG_ALPHA, pad=1.5,
                              boxstyle="round,pad=0.2"),
                )
                tick_texts.append(t)

        # --- Title ---
        if TITLE_FONT_SIZE > 0:
            ax.set_title(
                f"Metabolic Pathway Scores — {cell_type}",
                fontsize=TITLE_FONT_SIZE, fontweight="bold",
                pad=25, y=1.05,
            )

        # --- Legend ---
        leg = ax.legend(
            loc=LEGEND_LOCATION, bbox_to_anchor=LEGEND_BBOX,
            ncol=LEGEND_NCOL, fontsize=LEGEND_FONT_SIZE,
            frameon=False, handlelength=2.5, columnspacing=1.5,
        )
        for line in leg.get_lines():
            line.set_linewidth(LEGEND_LINE_WIDTH)

        return fig, ax, tick_texts

    # --- VERSION 1: WITH tick numbers on top ---
    fig1, ax1, _ = _build_radar(show_ticks=True)
    pdf_with = os.path.join(output_dir, f"spider_plot_{cell_type}_with_ticks.pdf")
    plt.tight_layout()
    fig1.savefig(pdf_with, format="pdf", dpi=DPI, bbox_inches="tight")
    plt.close(fig1)
    print(f"  Plot (with ticks) saved: {pdf_with}")

    # --- VERSION 2: WITHOUT any numbers inside ---
    fig2, ax2, _ = _build_radar(show_ticks=False)
    pdf_clean = os.path.join(output_dir, f"spider_plot_{cell_type}_clean.pdf")
    plt.tight_layout()
    fig2.savefig(pdf_clean, format="pdf", dpi=DPI, bbox_inches="tight")
    plt.close(fig2)
    print(f"  Plot (clean) saved: {pdf_clean}")


# ==============================================================================
# 3. GENERATE PLOTS
# ==============================================================================

for ct in CELL_TYPES_TO_PLOT:
    make_spider_plot(
        adata        = adata_gss,
        cell_type    = ct,
        pathway_map  = PATHWAY_DISPLAY,
        cond_order   = COND_ORDER,
        cond_palette = COND_PALETTE,
        cond_labels  = COND_LABELS,
        ct_col       = CELL_TYPE_COL,
        cond_col     = CONDITION_COL,
        samp_col     = SAMPLE_COL,
        output_dir   = OUTPUT_DIR,
    )

print("\nAll spider plots complete!")



[CM] 182612 cells


  Plot (with ticks) saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots/spider_plot_CM_with_ticks.pdf


  Plot (clean) saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots/spider_plot_CM_clean.pdf

All spider plots complete!


In [21]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from math import pi
import scanpy as sc

warnings.filterwarnings("ignore")

# ── GLOBAL FULL-FRAME PATCH ──────────────────────────────────────────────────
import matplotlib.figure as _mfig
if not getattr(_mfig.Figure, "_frame_patched", False):
    _orig_savefig = _mfig.Figure.savefig
    def _framed_savefig(self, *a, **k):
        for _ax in self.axes:
            if getattr(_ax, "name", "") == "rectilinear":
                for _s in _ax.spines.values():
                    _s.set_visible(True); _s.set_color("black"); _s.set_linewidth(1.1)
        return _orig_savefig(self, *a, **k)
    _mfig.Figure.savefig = _framed_savefig
    _mfig.Figure._frame_patched = True

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# --- A. Input / Output ---
INPUT_FILE = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/global_metabolic_gssa_preclinical.h5ad"
OUTPUT_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- B. Which cell type(s) to plot ---
CELL_TYPES_TO_PLOT = ["CM"]      # Change to ["CM", "EC", "FB"] etc.
CELL_TYPE_COL      = "cell_type" # Column in adata.obs
CONDITION_COL      = "Condition" # Column in adata.obs
SAMPLE_COL         = "Sample_ID" # Column in adata.obs

# --- C. Conditions to show (in this order in the legend) ---
# ---> OPTIONAL: Exclude any conditions you don't want to plot <---
EXCLUDE_CONDITIONS = ["Control"]

COND_ORDER = [
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
    "Control",
    "Prematurely-terminated",
]

# Filter out excluded conditions dynamically
COND_ORDER = [c for c in COND_ORDER if c not in EXCLUDE_CONDITIONS]

# --- D. Condition colors ---
COND_PALETTE = {
    "Control":                "#EFE48E",
    "3month-protocol":        "skyblue",
    "6month-protocol":        "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV":               "#F0C096",
    "Rejection":              "#E09E9E",
}

# --- E. Condition display labels (short names for legend) ---
COND_LABELS = {
    "Control":                "Control",
    "3month-protocol":        "3-month protocol",
    "6month-protocol":        "6-month protocol",
    "Prematurely-terminated": "Preterm",
    "PCMV/PRV":               "PCMV/PRV",
    "Rejection":              "Rejection",
}

# --- F. Pathway display names (order here = order on the plot, clockwise) ---
PATHWAY_DISPLAY = {
    "Lactate_metabolism":       "Lactate",
    "Ketone_metabolism":        "Ketone",
    "FAO":                      "FAO",
    "Metabolic_dysfunction":    "Metabolic\nDysfunction",
    "TCA_OXPHOS":               "TCA Cycle",
    "Amino_acid_metabolism":    "Amino Acid",
    "Carbohydrate_metabolism":  "Carbohydrate",
    "Glycolysis":               "Glycolysis",
}

# --- G. Radar visual parameters (Matched to Metabolomics Script) ---
FIGURE_SIZE         = (8, 8)        # (width, height) in inches
RADAR_LINE_WIDTH    = 3.0           # Thickness of each condition's polygon line
RADAR_FILL_ALPHA    = 0.15          # Transparency of the filled polygon area
RADAR_DOT_SIZE      = 60            # Size of dots at each vertex
BACKGROUND_COLOR    = "#FAFAFA"     # Subtle off-white background
GRID_COLOR          = "#E0E0E0"     # Color of the concentric grid circles
GRID_LINE_WIDTH     = 1.2           # Thickness of grid lines
GRID_LINE_STYLE     = "--"          # "-" solid, "--" dashed, ":" dotted
SPOKE_COLOR         = "#E0E0E0"     # Color of the radial spoke lines
SPOKE_LINE_WIDTH    = 1.2           # Thickness of spoke lines
LABEL_FONT_SIZE     = 11            # Font size for pathway labels around the edge
LABEL_FONT_WEIGHT   = "bold"        # Font weight for pathway labels
LABEL_PAD           = 25            # Distance of labels from center
TICK_FONT_SIZE      = 9             # Font size for the radial tick values
TICK_FONT_COLOR     = "#333333"     # Color for tick values
TICK_BG_ALPHA       = 0.7           # Background opacity behind tick numbers (0-1)
TICK_ANGLE_DEG      = 0             # Angle (degrees) at which tick numbers are placed
LEGEND_FONT_SIZE    = 11            # Font size for legend entries
LEGEND_LINE_WIDTH   = 3.0           # Thickness of lines in the legend
LEGEND_LOCATION     = "lower center"
LEGEND_NCOL         = 2             # Number of columns in legend
LEGEND_BBOX         = (0.5, -0.28)  # (x, y) — decrease y to push legend further down
TITLE_FONT_SIZE     = 14            # Title font size (set 0 to hide title)
DPI                 = 300           # Export resolution

# --- H. Global font settings ---
mpl.rcParams["pdf.fonttype"]    = 42
mpl.rcParams["ps.fonttype"]     = 42
mpl.rcParams["font.family"]     = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.weight"]     = "bold"
sc.settings.set_figure_params(dpi=DPI, dpi_save=DPI)


# ==============================================================================
# 2. SPIDER PLOT FUNCTION
# ==============================================================================

def make_spider_plot(adata, cell_type, pathway_map, cond_order, cond_palette,
                     cond_labels, ct_col, cond_col, samp_col, output_dir):
    """
    Generates a publication-quality radar/spider plot for one cell type.
    Saves TWO PDFs: one with tick numbers on top, one clean without numbers.
    The plotted values and similarities are written by the export cell at the end of the notebook.
    """

    # --- Subset to cell type ---
    mask = adata.obs[ct_col].astype(str) == cell_type
    ad = adata[mask].copy()
    print(f"\n[{cell_type}] {ad.n_obs} cells")

    # --- Build score columns ---
    score_stems = list(pathway_map.keys())
    score_cols  = [f"{s}_score" for s in score_stems]
    missing = [c for c in score_cols if c not in ad.obs.columns]
    if missing:
        print(f"  [WARN] Missing score columns: {missing}")
        score_cols  = [c for c in score_cols if c in ad.obs.columns]
        score_stems = [c.replace("_score", "") for c in score_cols]

    # --- Pseudobulk: median per sample, then median per condition ---
    df = ad.obs[score_cols + [samp_col, cond_col]].copy()
    df.columns = [c.replace("_score", "") if c.endswith("_score") else c for c in df.columns]

    sample_med = df.groupby(samp_col, observed=True)[score_stems].median()
    s2c = df.drop_duplicates(samp_col).set_index(samp_col)[cond_col]
    sample_med[cond_col] = s2c.loc[sample_med.index]

    cond_med = sample_med.groupby(cond_col, observed=True)[score_stems].median()

    # Filter to requested conditions that exist
    present = [c for c in cond_order if c in cond_med.index]
    cond_med = cond_med.loc[present]

    # Reorder columns to match pathway_map order
    ordered_stems = [s for s in score_stems if s in cond_med.columns]
    cond_med = cond_med[ordered_stems]

    # --- Radar geometry ---
    N = len(ordered_stems)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]

    labels = [pathway_map.get(s, s) for s in ordered_stems]

    # --- Radial tick values (auto from data range) ---
    all_vals = cond_med.values.flatten()
    v_min = np.floor(all_vals.min() * 20) / 20
    v_max = np.ceil(all_vals.max() * 20) / 20
    n_ticks = 5
    tick_vals = np.linspace(v_min, v_max, n_ticks)

    # ==========================================================================
    # Helper: build the radar plot (returns fig, ax, and the manual tick artists)
    # ==========================================================================
    def _build_radar(show_ticks: bool):
        fig, ax = plt.subplots(figsize=FIGURE_SIZE, subplot_kw=dict(polar=True))

        ax.set_theta_offset(pi / 2)
        ax.set_theta_direction(-1)

        # Pathway labels at each spoke
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, fontsize=LABEL_FONT_SIZE, fontweight=LABEL_FONT_WEIGHT)
        
        # Safely adjust the distance of labels from the circle using pad
        ax.tick_params(axis='x', pad=LABEL_PAD)

        # Radial axis
        ax.set_ylim(v_min - 0.02, v_max + 0.02)
        ax.set_yticks(tick_vals)
        ax.set_yticklabels([])  # always hide default labels; we draw our own

        # Fancier aesthetics: Subtle background and dashed y-grid
        ax.set_facecolor(BACKGROUND_COLOR)
        ax.yaxis.grid(True, color=GRID_COLOR, linewidth=GRID_LINE_WIDTH, linestyle=GRID_LINE_STYLE)
        ax.xaxis.grid(True, color=SPOKE_COLOR, linewidth=SPOKE_LINE_WIDTH)
        ax.spines["polar"].set_visible(False)

        # --- Plot each condition polygon ---
        for cond in present:
            values = cond_med.loc[cond].values.tolist()
            values += values[:1]

            color = cond_palette.get(cond, "gray")
            lbl   = cond_labels.get(cond, cond)

            # Plot line with round caps for smoother look
            ax.plot(angles, values, linewidth=RADAR_LINE_WIDTH, color=color, 
                    label=lbl, solid_capstyle="round", zorder=3)
            # Slightly higher alpha for richer fill
            ax.fill(angles, values, alpha=RADAR_FILL_ALPHA, color=color, zorder=2)
            # Fancier scatter points with crisp white borders
            ax.scatter(angles[:-1], values[:-1], s=RADAR_DOT_SIZE, color=color, 
                       edgecolors="white", linewidths=1.5, zorder=4)

        # --- Manually draw tick numbers ON TOP of polygons ---
        tick_texts = []
        if show_ticks:
            tick_angle_rad = np.deg2rad(TICK_ANGLE_DEG)
            for tv in tick_vals:
                t = ax.text(
                    tick_angle_rad, tv, f"{tv:.2f}",
                    fontsize=TICK_FONT_SIZE,
                    color=TICK_FONT_COLOR,
                    ha="center", va="center",
                    zorder=15,
                    bbox=dict(facecolor="white", edgecolor="none",
                              alpha=TICK_BG_ALPHA, pad=1.5,
                              boxstyle="round,pad=0.2"),
                )
                tick_texts.append(t)

        # --- Title ---
        if TITLE_FONT_SIZE > 0:
            ax.set_title(
                f"Metabolic Pathway Scores — {cell_type}",
                fontsize=TITLE_FONT_SIZE, fontweight="bold",
                pad=30, y=1.05,
            )

        # --- Legend ---
        leg = ax.legend(
            loc=LEGEND_LOCATION, bbox_to_anchor=LEGEND_BBOX,
            ncol=LEGEND_NCOL, fontsize=LEGEND_FONT_SIZE,
            frameon=False, handlelength=2.5, columnspacing=1.5,
        )
        for line in leg.get_lines():
            line.set_linewidth(LEGEND_LINE_WIDTH)

        return fig, ax, tick_texts

    # --- VERSION 1: WITH tick numbers on top ---
    fig1, ax1, _ = _build_radar(show_ticks=True)
    pdf_with = os.path.join(output_dir, f"spider_plot_{cell_type}_with_ticks.pdf")
    plt.tight_layout()
    fig1.savefig(pdf_with, format="pdf", dpi=DPI, bbox_inches="tight", transparent=True)
    plt.close(fig1)
    print(f"  Plot (with ticks) saved: {pdf_with}")

    # --- VERSION 2: WITHOUT any numbers inside ---
    fig2, ax2, _ = _build_radar(show_ticks=False)
    pdf_clean = os.path.join(output_dir, f"spider_plot_{cell_type}_clean.pdf")
    plt.tight_layout()
    fig2.savefig(pdf_clean, format="pdf", dpi=DPI, bbox_inches="tight", transparent=True)
    plt.close(fig2)
    print(f"  Plot (clean) saved: {pdf_clean}")


# ==============================================================================
# 3. GENERATE PLOTS
# ==============================================================================

# print("Loading GSS adata...")
# adata_gss = sc.read(INPUT_FILE)

for ct in CELL_TYPES_TO_PLOT:
    make_spider_plot(
        adata        = adata_gss,
        cell_type    = ct,
        pathway_map  = PATHWAY_DISPLAY,
        cond_order   = COND_ORDER,
        cond_palette = COND_PALETTE,
        cond_labels  = COND_LABELS,
        ct_col       = CELL_TYPE_COL,
        cond_col     = CONDITION_COL,
        samp_col     = SAMPLE_COL,
        output_dir   = OUTPUT_DIR,
    )

print("\nAll spider plots complete!")


[CM] 182612 cells


  Plot (with ticks) saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots/spider_plot_CM_with_ticks.pdf
  Plot (clean) saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots/spider_plot_CM_clean.pdf



All spider plots complete!


# Spider plot: all cell types pooled

In [22]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from math import pi
import scanpy as sc

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURATION (identical to the cardiomyocyte radar above, except the output folder)
# ==============================================================================
GLOBAL_SPIDER_DIR = os.path.join(WORKING_DIR, "spider_plots_global")
os.makedirs(GLOBAL_SPIDER_DIR, exist_ok=True)
GLOBAL_SPIDER_NAME = "all_celltypes"          # file-name tag
GLOBAL_SPIDER_TITLE = "all cell types"        # title tag
CONDITION_COL = "Condition"
SAMPLE_COL = "Sample_ID"

EXCLUDE_CONDITIONS = ["Control"]              # as in the cardiomyocyte radar; set [] to draw Control too
SPIDER_COND_ORDER = [c for c in ["3month-protocol", "6month-protocol", "PCMV/PRV", "Rejection",
                                 "Control", "Prematurely-terminated"] if c not in EXCLUDE_CONDITIONS]
SPIDER_PALETTE = {
    "Control":                "#EFE48E",
    "3month-protocol":        "skyblue",
    "6month-protocol":        "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV":               "#F0C096",
    "Rejection":              "#E09E9E",
}
SPIDER_LABELS = {
    "Control":                "Control",
    "3month-protocol":        "3-month protocol",
    "6month-protocol":        "6-month protocol",
    "Prematurely-terminated": "Preterm",
    "PCMV/PRV":               "PCMV/PRV",
    "Rejection":              "Rejection",
}
SPIDER_PATHWAYS = {                            # order here = order on the plot, clockwise
    "Lactate_metabolism":       "Lactate",
    "Ketone_metabolism":        "Ketone",
    "FAO":                      "FAO",
    "Metabolic_dysfunction":    "Metabolic\nDysfunction",
    "TCA_OXPHOS":               "TCA Cycle",
    "Amino_acid_metabolism":    "Amino Acid",
    "Carbohydrate_metabolism":  "Carbohydrate",
    "Glycolysis":               "Glycolysis",
}
SPIDER_STYLE = dict(
    figure_size=(8, 8), line_width=3.0, fill_alpha=0.15, dot_size=60, background="#FAFAFA",
    grid_color="#E0E0E0", grid_lw=1.2, grid_ls="--", spoke_color="#E0E0E0", spoke_lw=1.2,
    label_fs=11, label_weight="bold", label_pad=25, tick_fs=9, tick_color="#333333", tick_bg_alpha=0.7,
    tick_angle_deg=0, legend_fs=11, legend_lw=3.0, legend_loc="lower center", legend_ncol=2,
    legend_bbox=(0.5, -0.28), title_fs=14, dpi=300,
)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.weight"] = "bold"

# ==============================================================================
# 2. VALUES: median per heart over all nuclei, then median per condition
# ==============================================================================
stems = [s for s in SPIDER_PATHWAYS if f"{s}_score" in adata_gss.obs.columns]
d_all = adata_gss.obs[[f"{s}_score" for s in stems] + [SAMPLE_COL, CONDITION_COL]].copy()
d_all.columns = stems + [SAMPLE_COL, CONDITION_COL]
heart_med = d_all.groupby(SAMPLE_COL, observed=True)[stems].median()
heart_med[CONDITION_COL] = (d_all.drop_duplicates(SAMPLE_COL).set_index(SAMPLE_COL)[CONDITION_COL]
                            .astype(str).loc[heart_med.index].values)
global_radar = heart_med.groupby(CONDITION_COL, observed=True)[stems].median()
spider_present = [c for c in SPIDER_COND_ORDER if c in global_radar.index]
global_radar = global_radar.loc[spider_present, stems]

# ==============================================================================
# 3. DRAW (same geometry and styling as the cardiomyocyte radar)
# ==============================================================================
s = SPIDER_STYLE
N = len(stems)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]
labels = [SPIDER_PATHWAYS[st] for st in stems]
all_vals = global_radar.values.flatten()
v_min = np.floor(all_vals.min() * 20) / 20
v_max = np.ceil(all_vals.max() * 20) / 20
tick_vals = np.linspace(v_min, v_max, 5)

def _build_global_radar(show_ticks):
    fig, ax = plt.subplots(figsize=s["figure_size"], subplot_kw=dict(polar=True))
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=s["label_fs"], fontweight=s["label_weight"])
    ax.tick_params(axis="x", pad=s["label_pad"])
    ax.set_ylim(v_min - 0.02, v_max + 0.02)
    ax.set_yticks(tick_vals)
    ax.set_yticklabels([])
    ax.set_facecolor(s["background"])
    ax.yaxis.grid(True, color=s["grid_color"], linewidth=s["grid_lw"], linestyle=s["grid_ls"])
    ax.xaxis.grid(True, color=s["spoke_color"], linewidth=s["spoke_lw"])
    ax.spines["polar"].set_visible(False)
    for cond in spider_present:
        values = global_radar.loc[cond].values.tolist()
        values += values[:1]
        color = SPIDER_PALETTE.get(cond, "gray")
        ax.plot(angles, values, linewidth=s["line_width"], color=color, label=SPIDER_LABELS.get(cond, cond),
                solid_capstyle="round", zorder=3)
        ax.fill(angles, values, alpha=s["fill_alpha"], color=color, zorder=2)
        ax.scatter(angles[:-1], values[:-1], s=s["dot_size"], color=color, edgecolors="white",
                   linewidths=1.5, zorder=4)
    if show_ticks:
        for tv in tick_vals:
            ax.text(np.deg2rad(s["tick_angle_deg"]), tv, f"{tv:.2f}", fontsize=s["tick_fs"], color=s["tick_color"],
                    ha="center", va="center", zorder=15,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=s["tick_bg_alpha"], pad=1.5,
                              boxstyle="round,pad=0.2"))
    if s["title_fs"] > 0:
        ax.set_title(f"Metabolic Pathway Scores — {GLOBAL_SPIDER_TITLE}", fontsize=s["title_fs"],
                     fontweight="bold", pad=30, y=1.05)
    leg = ax.legend(loc=s["legend_loc"], bbox_to_anchor=s["legend_bbox"], ncol=s["legend_ncol"],
                    fontsize=s["legend_fs"], frameon=False, handlelength=2.5, columnspacing=1.5)
    for line in leg.get_lines():
        line.set_linewidth(s["legend_lw"])
    return fig

for show_ticks, tag in ((True, "with_ticks"), (False, "clean")):
    fig = _build_global_radar(show_ticks)
    out_pdf = os.path.join(GLOBAL_SPIDER_DIR, f"spider_plot_{GLOBAL_SPIDER_NAME}_{tag}.pdf")
    plt.tight_layout()
    fig.savefig(out_pdf, format="pdf", dpi=s["dpi"], bbox_inches="tight", transparent=True)
    plt.close(fig)
    print(f"Saved: {out_pdf}")

print(f"\n{d_all.shape[0]:,} nuclei, {len(heart_med)} hearts; values plotted (median of the per-heart medians):")
print(global_radar.round(4).to_string())


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots_global/spider_plot_all_celltypes_with_ticks.pdf
Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/Metabolic/results/spider_plots_global/spider_plot_all_celltypes_clean.pdf

233,200 nuclei, 26 hearts; values plotted (median of the per-heart medians):
                        Lactate_metabolism  Ketone_metabolism     FAO  Metabolic_dysfunction  TCA_OXPHOS  Amino_acid_metabolism  Carbohydrate_metabolism  Glycolysis
Condition                                                                                                                                                           
3month-protocol                    -0.1843             0.1735  0.1076                -0.0516     -0.0377                 0.0130                   0.1877     -0.1048
6month-protocol                    -0.0828             0.1010  0.0330                -0.0027     -0.0784      

# exporting and stitching CSVs together to make 1 giant excel file

In [23]:
from itertools import combinations
from scipy import stats
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# --- 1. SETTINGS ---
CONDITIONS = ["Control", "Prematurely-terminated", "3month-protocol", "6month-protocol", "PCMV/PRV", "Rejection"]
REF_COND = "3month-protocol"                     # reference of the per-cell-type tests (as in the boxplots)
RADAR_PROFILES = {"CM": "Radar_plot_CM", "All nuclei": "Radar_plot_all_nuclei"}   # radar plots: profile -> sheet
POOLED_COMPARISONS = ["Prematurely-terminated", "6month-protocol", "Rejection", "PCMV/PRV"]  # all-nuclei tests vs 3month
RADAR_CONDITIONS = ["3month-protocol", "6month-protocol", "PCMV/PRV", "Rejection", "Prematurely-terminated"]  # Control not shown
RADAR_AXES = ["Lactate_metabolism", "Ketone_metabolism", "FAO", "Metabolic_dysfunction", "TCA_OXPHOS",
              "Amino_acid_metabolism", "Carbohydrate_metabolism", "Glycolysis"]
WORKBOOK = os.path.join(WORKING_DIR, "Supplementary_Table_metabolic_gene_set_scores.xlsx")

PATHS = list(pathways.keys())
LABEL = {p: p.replace("_", " ").replace("TCA OXPHOS", "TCA/OXPHOS") for p in PATHS}
P_COLS = [LABEL[p] for p in PATHS]
CELL_TYPES = sorted(adata_gss.obs["cell_type"].dropna().unique().astype(str).tolist())

obs = adata_gss.obs[["Sample_ID", "Condition", "cell_type", "Survival"] + [f"{p}_score" for p in PATHS]].copy()
obs = obs.rename(columns={f"{p}_score": LABEL[p] for p in PATHS})
heart_info = obs.drop_duplicates("Sample_ID").set_index("Sample_ID")[["Condition", "Survival"]]
cond_rank = {c: i for i, c in enumerate(CONDITIONS)}
ct_rank = {c: i for i, c in enumerate(["All nuclei"] + CELL_TYPES)}

# --- 2. GENE SETS (genes as scored, after the expression filter) ---
gene_sets = pd.DataFrame([{"Pathway": LABEL[p], "Gene": g, "Genes in pathway (n)": len(pathways[p]),
                           "Also in": "; ".join(LABEL[q] for q in PATHS if q != p and g in pathways[q])}
                          for p in PATHS for g in pathways[p]])
gene_overlap = pd.DataFrame([[len(set(pathways[a]) & set(pathways[b])) for b in PATHS] for a in PATHS],
                            index=P_COLS, columns=P_COLS).rename_axis("Pathway").reset_index()

# --- 3. SCORES PER HEART (median over the heart's nuclei) ---
def heart_medians(df, cell_type):
    g = df.groupby("Sample_ID", observed=True)
    med = g[P_COLS].median()
    out = pd.DataFrame({"Cell type": cell_type, "Heart": med.index.astype(str),
                        "Condition": heart_info.loc[med.index, "Condition"].astype(str).values,
                        "Survival (days)": heart_info.loc[med.index, "Survival"].values,
                        "Nuclei (n)": g.size().loc[med.index].values})
    return pd.concat([out, med.reset_index(drop=True)], axis=1)

per_heart = pd.concat([heart_medians(obs, "All nuclei")] +
                      [heart_medians(obs[obs["cell_type"].astype(str) == ct], ct) for ct in CELL_TYPES], ignore_index=True)
per_heart = per_heart.sort_values(["Cell type", "Condition", "Heart"],
                                  key=lambda s: s.map(ct_rank) if s.name == "Cell type" else (s.map(cond_rank) if s.name == "Condition" else s)
                                  ).reset_index(drop=True)

# --- 4. SCORES PER CONDITION (median and mean of the per-heart medians) ---
rows = []
for ct in ["All nuclei"] + CELL_TYPES:
    d = per_heart[per_heart["Cell type"] == ct]
    for cond in [c for c in CONDITIONS if c in set(d["Condition"])]:
        dc = d[d["Condition"] == cond]
        for summary, how in (("Median of heart medians", "median"), ("Mean of heart medians", "mean")):
            rows.append({"Cell type": ct, "Condition": cond, "Summary": summary, "Hearts (n)": len(dc), **dc[P_COLS].agg(how).to_dict()})
per_cond = pd.DataFrame(rows)

# --- 5. SIMILARITY BETWEEN CONDITION PROFILES ---
# (a) all nuclei: Pearson r over the eight condition medians, two-sided test; BH across the five conditions compared with Control
glob_med = per_cond[(per_cond["Cell type"] == "All nuclei") & (per_cond["Summary"] == "Median of heart medians")].set_index("Condition")[P_COLS]
conds = list(glob_med.index)
sim = []
for i, a in enumerate(conds):
    for b in conds[i + 1:]:
        r, p = stats.pearsonr(glob_med.loc[a], glob_med.loc[b])
        sim.append({"Profile": "All nuclei, condition medians", "Measure": "Pearson r", "Condition A": a, "Condition B": b,
                    "Value": r, "Pathways (n)": len(P_COLS), "P": p})
sim = pd.DataFrame(sim)
vs_ctrl = sim["Condition A"].eq("Control")
sim["FDR"] = np.nan
sim.loc[vs_ctrl, "FDR"] = multipletests(sim.loc[vs_ctrl, "P"], method="fdr_bh")[1]

# (b) exact label permutation: is a transplanted condition closer to Control than any set of transplanted hearts of its size?
hearts_all = per_heart[per_heart["Cell type"] == "All nuclei"]
ctrl = glob_med.loc["Control"].values
ctrl_c = ctrl - ctrl.mean()
graft = hearts_all[hearts_all["Condition"] != "Control"]
X = graft[P_COLS].values

def r_to_ctrl(profiles):
    pc = profiles - profiles.mean(axis=1, keepdims=True)
    return (pc @ ctrl_c) / (np.linalg.norm(pc, axis=1) * np.linalg.norm(ctrl_c))

for col in ("Permutation sets (n)", "Sets as close to Control (n)", "Permutation P"):
    sim[col] = np.nan
for arm in [c for c in conds if c != "Control"]:
    is_arm = (graft["Condition"] == arm).values
    r_obs = r_to_ctrl(np.median(X[is_arm], axis=0)[None, :])[0]
    idx = np.array(list(combinations(range(len(graft)), int(is_arm.sum()))))
    r_perm = r_to_ctrl(np.median(X[idx], axis=1))
    n_close = int((r_perm >= r_obs - 1e-12).sum())
    sel = (sim["Condition A"] == "Control") & (sim["Condition B"] == arm)
    sim.loc[sel, ["Permutation sets (n)", "Sets as close to Control (n)", "Permutation P"]] = [len(r_perm), n_close, n_close / len(r_perm)]

# (c) radar plots (cardiomyocytes, all nuclei): Pearson r and cosine similarity between the condition medians (descriptive, as plotted)
rad_rows, radar_sheets = [], {}
for ct, sheet_name in RADAR_PROFILES.items():
    radar = per_cond[(per_cond["Cell type"] == ct) & (per_cond["Summary"] == "Median of heart medians")
                     & per_cond["Condition"].isin(RADAR_CONDITIONS)].set_index("Condition").loc[RADAR_CONDITIONS]
    radar_vals = radar[[LABEL[p] for p in RADAR_AXES]]
    for i, a in enumerate(RADAR_CONDITIONS):
        for b in RADAR_CONDITIONS[i + 1:]:
            va, vb = radar_vals.loc[a].values, radar_vals.loc[b].values
            for measure, v in (("Pearson r", np.corrcoef(va, vb)[0, 1]),
                               ("Cosine similarity", va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))):
                rad_rows.append({"Profile": f"{ct} radar plot, condition medians", "Measure": measure,
                                 "Condition A": a, "Condition B": b, "Value": v, "Pathways (n)": len(RADAR_AXES)})
    radar_sheets[sheet_name] = radar[["Hearts (n)"]].join(radar_vals).reset_index()
sim = pd.concat([sim, pd.DataFrame(rad_rows)], ignore_index=True)
if "global_radar" in globals():   # the all-nuclei radar cell above drew exactly these values
    drawn = global_radar.rename(columns=LABEL)[[LABEL[p] for p in RADAR_AXES]]
    table = radar_sheets["Radar_plot_all_nuclei"].set_index("Condition").loc[drawn.index, drawn.columns]
    assert np.allclose(drawn.values, table.values), "all-nuclei radar values differ from the plot"

# --- 6. TESTS AGAINST 3MONTH-PROTOCOL (as in the significance boxplots) ---
# per cell type: every other condition present; all nuclei pooled: the four transplanted arms in POOLED_COMPARISONS
def test_block(d, ct, comparisons):
    block = []
    for p in P_COLS:
        ref = d.loc[d["Condition"] == REF_COND, p].dropna().values
        for comp in comparisons:
            cmp_ = d.loc[d["Condition"] == comp, p].dropna().values
            pval = np.nan
            if len(ref) and len(cmp_):
                try:
                    pval = mannwhitneyu(cmp_, ref, alternative="two-sided").pvalue
                except Exception:
                    pass
            m_ref = np.mean(ref) if len(ref) else np.nan
            m_cmp = np.mean(cmp_) if len(cmp_) else np.nan
            block.append({"Cell type": ct, "Reference": REF_COND, "Compared condition": comp, "Pathway": p,
                          "Hearts, reference (n)": len(ref), "Hearts, compared (n)": len(cmp_),
                          "Mean, reference": m_ref, "Mean, compared": m_cmp, "Difference": m_cmp - m_ref, "P": pval})
    block = pd.DataFrame(block)
    block["FDR"] = np.nan
    for comp in block["Compared condition"].unique():
        m = (block["Compared condition"] == comp) & block["P"].notna()
        if m.any():
            block.loc[m, "FDR"] = multipletests(block.loc[m, "P"], method="fdr_bh")[1]
    return block

tests = []
for ct in CELL_TYPES:
    d = per_heart[per_heart["Cell type"] == ct]
    present = [c for c in CONDITIONS if c in set(d["Condition"])]
    if REF_COND in present:
        tests.append(test_block(d, ct, [c for c in present if c != REF_COND]))
pooled_block = test_block(per_heart[per_heart["Cell type"] == "All nuclei"], "All nuclei", POOLED_COMPARISONS)
if "pooled_tests" in globals():   # the pooled boxplot cell above starred exactly these results
    chk = pooled_tests.assign(Pathway=pooled_tests["Pathway"].map(LABEL)).set_index(["Compared condition", "Pathway"])
    mine = pooled_block.set_index(["Compared condition", "Pathway"]).loc[chk.index]
    assert np.allclose(chk[["P", "FDR"]].values, mine[["P", "FDR"]].values), "pooled tests differ from the boxplots"
tests = pd.concat(tests + [pooled_block], ignore_index=True)
tests["Mark"] = np.select([tests["FDR"] < 0.001, tests["FDR"] < 0.01, tests["FDR"] < 0.05], ["***", "**", "*"], "")
tests = tests.sort_values(["Cell type", "Compared condition", "Pathway"],
                          key=lambda s: s.map(cond_rank) if s.name == "Compared condition"
                          else (s.map({l: i for i, l in enumerate(P_COLS)}) if s.name == "Pathway" else s)).reset_index(drop=True)

# --- 7. ROUND FOR OUTPUT AND WRITE THE MASTER CSVs ---
def rounded(df, cols, k=4):
    out = df.copy()
    out[cols] = out[cols].astype(float).round(k)
    return out

per_heart_out = rounded(per_heart, P_COLS)
per_cond_out = rounded(per_cond, P_COLS)
sim_out = rounded(sim, ["Value"])
tests_out = rounded(tests, ["Mean, reference", "Mean, compared", "Difference"])
radar_out = {k: rounded(v, [LABEL[p] for p in RADAR_AXES]) for k, v in radar_sheets.items()}

gene_sets.to_csv("MASTER_metabolic_gene_sets.csv", index=False)
per_heart_out.to_csv("MASTER_metabolic_scores_per_heart.csv", index=False)
per_cond_out.to_csv("MASTER_metabolic_scores_per_condition.csv", index=False)
sim_out.to_csv("MASTER_metabolic_condition_similarity.csv", index=False)
tests_out.to_csv("MASTER_metabolic_tests_vs_3month.csv", index=False)

# --- 8. ONE WORKBOOK: README + DATA SHEETS ---
about = [
    "Scoring: counts per nucleus were normalised to 10,000 and log1p-transformed, and each pathway was scored per nucleus with scanpy "
    "score_genes (mean expression of the pathway genes minus that of a random reference gene set matched for expression). Only genes "
    "with a mean expression of at least 0.0125 in at least one condition were kept; the genes used are listed in Gene_sets.",
    "Per heart: median score over all nuclei of the heart, or over its nuclei of one cell type.",
    "Per condition: median of the per-heart medians (used for the condition heatmap, the correlation matrix and the radar plots) or "
    "mean of the per-heart medians (used for the per-cell-type heatmaps).",
    "Similarity: Pearson r between condition profiles over the eight pathway medians, two-sided test (df = 6); Benjamini-Hochberg FDR "
    "across the five conditions compared with Control (other pairs: r and P only). Exact permutation: for each transplanted condition against Control, every set of transplanted hearts "
    "of the same size was summarised in the same way; P is the share of sets whose profile correlates with Control at least as strongly "
    "as the condition itself. Radar plots: Pearson r and cosine similarity between the condition medians of the cardiomyocyte and "
    "of the all-nuclei radar (descriptive).",
    "Tests: per cell type, the per-heart medians of each condition against 3month-protocol; all nuclei pooled (cell type 'All "
    "nuclei'), the per-heart medians over all nuclei of Prematurely-terminated, 6month-protocol, Rejection and PCMV/PRV against "
    "3month-protocol (Control not tested). Two-sided Mann-Whitney U test; Benjamini-Hochberg FDR across the eight pathways within "
    "each cell type and comparison. Mark: * FDR < 0.05, ** < 0.01, *** < 0.001.",
    "Hearts: " + ", ".join(f"{c} {int((hearts_all['Condition'] == c).sum())}" for c in CONDITIONS)
    + ". Survival is given as recorded in the sample metadata.",
]
sheets = [
    dict(name="Gene_sets", df=gene_sets, what="genes of each pathway, as scored",
         cols={"Also in": "other pathways that contain the gene"}),
    dict(name="Gene_overlap", df=gene_overlap, what="genes shared between pathways (diagonal: genes per pathway)"),
    dict(name="Heart_scores", df=per_heart_out[per_heart_out["Cell type"] == "All nuclei"].drop(columns="Cell type"),
         what="median score of each pathway per heart, all nuclei",
         cols={"Nuclei (n)": "nuclei of the heart behind the median"}),
    dict(name="Heart_scores_by_cell_type", df=per_heart_out[per_heart_out["Cell type"] != "All nuclei"],
         what="median score of each pathway per heart and cell type",
         cols={"Nuclei (n)": "nuclei of that cell type in the heart"}),
    dict(name="Condition_scores", df=per_cond_out[per_cond_out["Cell type"] == "All nuclei"].drop(columns="Cell type"),
         what="median and mean of the per-heart medians per condition, all nuclei",
         cols={"Summary": "median or mean over the hearts of the condition", "Hearts (n)": "hearts of the condition"}),
    dict(name="Condition_scores_by_cell_type", df=per_cond_out[per_cond_out["Cell type"] != "All nuclei"],
         what="median and mean of the per-heart medians per condition and cell type",
         cols={"Hearts (n)": "hearts of the condition with nuclei of that cell type"}),
    dict(name="Condition_similarity", df=sim_out,
         what="similarity between condition profiles: Pearson r with P (all nuclei) and the spider similarities (CMs, all nuclei)",
         cols={"Value": "Pearson r or cosine similarity", "P": "two-sided Pearson test (all-nuclei profiles only)",
               "FDR": "Benjamini-Hochberg across the five conditions compared with Control",
               "Permutation P": "share of equally sized sets of transplanted hearts at least as close to Control"}),
    dict(name="Radar_plot_CM", df=radar_out["Radar_plot_CM"],
         what="values drawn in the cardiomyocyte radar plot (median of the per-heart medians)"),
    dict(name="Radar_plot_all_nuclei", df=radar_out["Radar_plot_all_nuclei"],
         what="values drawn in the all-nuclei radar plot (median of the per-heart medians over all nuclei)"),
    dict(name="Tests_vs_3month", df=tests_out,
         what="each condition against 3month-protocol per cell type and pathway, and the four transplanted arms pooled over all nuclei",
         cols={"Difference": "mean compared minus mean reference (means of the per-heart medians)",
               "P": "two-sided Mann-Whitney U test", "FDR": "Benjamini-Hochberg across the eight pathways of the comparison",
               "Mark": "* FDR < 0.05, ** < 0.01, *** < 0.001"}),
]
title = "Metabolic gene-set scores of pig-to-baboon cardiac xenografts snRNA-seq"
readme = [[title, "", ""], ["", "", ""]] + [[t, "", ""] for t in about] + [["", "", ""], ["Sheet", "Rows x columns", "Contents"]]
readme += [[s["name"], f"{len(s['df']):,} x {s['df'].shape[1]}", s["what"]] for s in sheets]
readme += [["", "", ""], ["Sheet", "Column", "Meaning"]] + [[s["name"], c, m] for s in sheets for c, m in s.get("cols", {}).items()]

with pd.ExcelWriter(WORKBOOK, engine="openpyxl") as xw:
    pd.DataFrame(readme).to_excel(xw, sheet_name="README", index=False, header=False)
    for s in sheets:
        s["df"].to_excel(xw, sheet_name=s["name"], index=False)
    wb = xw.book
    ws = wb["README"]
    ws["A1"].font = Font(bold=True, size=12)
    for r in ws.iter_rows():
        if r[0].value == "Sheet":
            for c in r: c.font = Font(bold=True)
        if r[2].value: r[2].alignment = Alignment(wrap_text=True, vertical="top")
    for col, w in (("A", 34), ("B", 30), ("C", 110)):
        ws.column_dimensions[col].width = w
    for s in sheets:
        ws = wb[s["name"]]
        ws.freeze_panes = "A2"
        for c in ws[1]: c.font = Font(bold=True)
        for j, col in enumerate(s["df"].columns, start=1):
            w = max([len(str(col))] + [len(str(v)) for v in s["df"][col].head(300).tolist()])
            ws.column_dimensions[get_column_letter(j)].width = min(max(w + 2, 8), 60)

# read every sheet back and compare with what was written
for s in sheets:
    back = pd.read_excel(WORKBOOK, sheet_name=s["name"])
    assert back.shape == s["df"].shape, f"{s['name']}: shape changed on writing"
    for c in s["df"].columns:
        a, b = s["df"][c].reset_index(drop=True), back[c]
        same = np.allclose(a.astype(float), b.astype(float), equal_nan=True) if pd.api.types.is_numeric_dtype(a) \
            else (a.fillna("").astype(str) == b.fillna("").astype(str)).all()
        assert same, f"{s['name']}: column {c} differs after writing"

print(f"Workbook: {os.path.basename(WORKBOOK)} ({len(sheets)} sheets + README, read back and identical)")
print(f"Hearts {len(hearts_all)}, cell types {len(CELL_TYPES)}, tests {len(tests)} ({int((tests['FDR'] < 0.05).sum())} at FDR < 0.05)")
pb = tests[tests["Cell type"] == "All nuclei"]
print(f"All nuclei pooled: {len(pb)} tests, {int((pb['FDR'] < 0.05).sum())} at FDR < 0.05")
print(sim_out[sim_out["Condition A"].eq("Control") & sim_out["Profile"].str.startswith("All")]
      [["Condition B", "Value", "P", "FDR", "Permutation P"]].to_string(index=False))

Workbook: Supplementary_Table_metabolic_gene_set_scores.xlsx (10 sheets + README, read back and identical)
Hearts 26, cell types 10, tests 360 (18 at FDR < 0.05)
All nuclei pooled: 32 tests, 4 at FDR < 0.05
           Condition B  Value        P      FDR  Permutation P
Prematurely-terminated 0.8756 0.004379 0.010948       0.294156
       3month-protocol 0.9257 0.000970 0.004851       0.034941
       6month-protocol 0.6382 0.088624 0.147706       0.857826
              PCMV/PRV 0.5636 0.145692 0.165962       0.865584
             Rejection 0.5412 0.165962 0.165962       0.949913
